# 🧥 FashionVision v2 — Real Clothing Classifier (MobileNetV2)

This notebook runs the **complete v2 training pipeline** on Google Colab GPU:

1. **Setup** — Install dependencies & clone repo
2. **Data** — Download the Kaggle `agrigorev/clothing-dataset-full` dataset (~5,000 real photos)
3. **Prepare** — Filter, split (70/15/15), resize to 224×224
4. **Train** — Two-stage transfer learning (frozen head → fine-tune top 30 layers)
5. **Evaluate** — Metrics, confusion matrix, misclassified examples on the test split
6. **Export** — Download `models_v2/` and `results_v2/` as a zip

> ⚡ **Runtime**: Set to **GPU** via `Runtime > Change runtime type > T4 GPU`

---
## 1. Setup Environment

In [ ]:
# Verify GPU is available
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU detected: {[g.name for g in gpus]}")
else:
    print("⚠️ No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Install required packages (kagglehub for dataset download)
!pip install -q kagglehub scikit-learn matplotlib pillow pandas

In [ ]:
# Clone your repository
import os

REPO_URL = "https://github.com/AdityaB197/FashionVision.git"
REPO_DIR = "/content/FashionVision"

if os.path.exists(REPO_DIR):
    print(f"Repo already cloned at {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}
    print(f"Cloned to {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Add project root to Python path
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verify imports work
from src.real_data import load_datasets, compute_class_weights
from src.real_model import build_real_model, enable_fine_tuning
from src.real_inference import load_real_artifacts, prepare_real_image, predict_top_k_real
from src.reporting import compute_metrics, plot_confusion_matrix, save_json
print("✅ All v2 modules imported successfully")

---
## 2. Download Dataset from Kaggle

You need Kaggle credentials. Two options:
- **Option A (Recommended)**: Use Colab Secrets — add `KAGGLE_USERNAME` and `KAGGLE_KEY` in the 🔑 sidebar
- **Option B**: Run `kagglehub.login()` and enter credentials interactively

In [ ]:
# Option A: Load from Colab Secrets (uncomment if using secrets)
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("✅ Kaggle credentials loaded from Colab Secrets")
except Exception:
    print("⚠️ Could not load Colab Secrets. Will try kagglehub.login() instead.")
    import kagglehub
    kagglehub.login()

In [ ]:
# Download the dataset (~7 GB, takes a few minutes)
import kagglehub

print("Downloading agrigorev/clothing-dataset-full ...")
raw_dataset_path = kagglehub.dataset_download("agrigorev/clothing-dataset-full")
print(f"✅ Dataset downloaded to: {raw_dataset_path}")

In [ ]:
# Quick look at the raw dataset structure
import subprocess
result = subprocess.run(['find', raw_dataset_path, '-maxdepth', '2', '-type', 'f', '-name', '*.csv'],
                       capture_output=True, text=True)
print("CSV files found:")
print(result.stdout)

result2 = subprocess.run(['find', raw_dataset_path, '-maxdepth', '2', '-type', 'd'],
                        capture_output=True, text=True)
print("Directories:")
print(result2.stdout)

---
## 3. Prepare Data (Filter, Split, Resize)

Runs `scripts/prepare_real_data.py` which:
- Filters out non-clothing / under-represented classes
- Splits 70% train / 15% val / 15% test (group-aware by sender_id when possible)
- Resizes all images to 224×224 with LANCZOS interpolation

In [ ]:
PREPARED_DIR = "data/clothing_prepared"

!python scripts/prepare_real_data.py \
    --source-dir "{raw_dataset_path}" \
    --output-dir "{PREPARED_DIR}" \
    --min-images-per-class 100 \
    --img-size 224 \
    --seed 42

In [ ]:
# Verify the prepared data structure
import json
from pathlib import Path

summary_path = Path(PREPARED_DIR) / "split_summary.json"
with open(summary_path) as f:
    split_summary = json.load(f)

print(f"Classes ({len(split_summary['class_names'])}): {split_summary['class_names']}")
print(f"\nImages per split:")
for split_name, counts in split_summary['counts_per_split'].items():
    total = sum(counts.values())
    print(f"  {split_name}: {total} images")
print(f"\nTotal: {split_summary['total_images_processed']} images")
print(f"Split method: {split_summary['split_method']}")

In [ ]:
# Visualize a few sample images from the prepared dataset
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

train_dir = Path(PREPARED_DIR) / "train"
class_dirs = sorted([d for d in train_dir.iterdir() if d.is_dir()])

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, cls_dir in zip(axes.flat, class_dirs[:10]):
    img_files = list(cls_dir.glob("*.jpg"))[:1]
    if img_files:
        img = Image.open(img_files[0])
        ax.imshow(img)
        ax.set_title(cls_dir.name, fontsize=10)
    ax.axis("off")

plt.suptitle("Sample Training Images (1 per class)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 4. Train Model (Two-Stage Transfer Learning)

- **Stage 1**: MobileNetV2 base frozen → train classification head (Adam LR=1e-3, up to 10 epochs)
- **Stage 2**: Unfreeze top 30 layers → fine-tune (Adam LR=1e-5, up to 10 more epochs)
- Early stopping with patience=4 restores best weights automatically

In [ ]:
# Full training (adjust epochs as needed)
!python scripts/train_real.py \
    --data-dir "{PREPARED_DIR}" \
    --output-dir "." \
    --epochs-head 10 \
    --epochs-finetune 10 \
    --batch-size 32 \
    --patience 4 \
    --seed 42

In [ ]:
# Verify training artifacts were created
from pathlib import Path

models_v2 = Path("models_v2")
results_v2 = Path("results_v2")

print("models_v2/ contents:")
for f in sorted(models_v2.iterdir()):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name:30s}  {size_mb:.2f} MB")

print(f"\nresults_v2/ contents:")
for f in sorted(results_v2.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:30s}  {size_kb:.1f} KB")

---
## 5. Evaluate on Test Split

In [ ]:
# Run the evaluation script on the held-out test split
!python scripts/evaluate_real.py \
    --data-dir "{PREPARED_DIR}" \
    --output-dir "."

In [ ]:
# Display metrics
import json

metrics_path = results_v2 / "metrics.json"
with open(metrics_path) as f:
    metrics = json.load(f)

print("="*50)
print(f"  Test Accuracy:  {metrics['accuracy']:.4f}")
print(f"  Macro F1:       {metrics['macro_f1']:.4f}")
print(f"  Weighted F1:    {metrics['weighted_f1']:.4f}")
print("="*50)

In [ ]:
# Display per-class classification report
import pandas as pd

clf_path = results_v2 / "classification_report.json"
with open(clf_path) as f:
    clf_report = json.load(f)

# Convert to DataFrame for pretty display
report_df = pd.DataFrame(clf_report).T
report_df = report_df.round(3)
display(report_df)

In [ ]:
# Display confusion matrix
from IPython.display import display as ipy_display, Image as IPyImage

cm_path = results_v2 / "confusion_matrix.png"
if cm_path.exists():
    ipy_display(IPyImage(filename=str(cm_path), width=700))
else:
    print("Confusion matrix image not found.")

In [ ]:
# Display training curves
curves_path = results_v2 / "training_curves.png"
if curves_path.exists():
    ipy_display(IPyImage(filename=str(curves_path), width=700))
else:
    print("Training curves image not found.")

In [ ]:
# Display misclassified examples
misc_path = results_v2 / "misclassified_examples.png"
if misc_path.exists():
    ipy_display(IPyImage(filename=str(misc_path), width=700))
else:
    print("Misclassified examples image not found.")

---
## 6. Quick Inference Sanity Check

In [ ]:
# Run the inference sanity check script
!python scripts/check_real_inference.py \
    --model-dir models_v2 \
    --data-dir "{PREPARED_DIR}"

In [ ]:
# Interactive: test on a random test image with visualization
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

model, class_names, metadata = load_real_artifacts("models_v2")
print(f"Model loaded: {metadata.get('model_name', 'v2')}")
print(f"Classes: {class_names}\n")

# Pick a random test image
test_dir = Path(PREPARED_DIR) / "test"
all_test_images = list(test_dir.rglob("*.jpg"))
sample_path = np.random.choice(all_test_images)
true_label = sample_path.parent.name

pil_img = Image.open(sample_path)
x = prepare_real_image(pil_img)
result = predict_top_k_real(model, x, class_names, k=5)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.imshow(pil_img)
ax1.set_title(f"True: {true_label}\nPredicted: {result['predicted_class']} ({result['confidence']:.1%})",
              fontsize=11)
ax1.axis("off")

top_k = result['top_k']
names = [t['class_name'] for t in top_k]
scores = [t['score'] for t in top_k]
colors = ['#2ecc71' if n == true_label else '#e74c3c' for n in names]
ax2.barh(names[::-1], scores[::-1], color=colors[::-1])
ax2.set_xlabel("Confidence")
ax2.set_title("Top-5 Predictions")
ax2.set_xlim(0, 1)

plt.tight_layout()
plt.show()

---
## 7. Download Trained Artifacts

Downloads `models_v2/` and `results_v2/` as a single zip file.
After downloading, extract into your local project root.

In [ ]:
import shutil
from google.colab import files

# Create zip archive
zip_name = "fashionvision_v2_artifacts"
zip_path = f"/content/{zip_name}"

# Remove old zip if exists
if os.path.exists(f"{zip_path}.zip"):
    os.remove(f"{zip_path}.zip")

# Create a staging directory with both folders
staging = Path("/content/staging_v2")
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir()

shutil.copytree("models_v2", staging / "models_v2")
shutil.copytree("results_v2", staging / "results_v2")

# Zip it
shutil.make_archive(zip_path, 'zip', staging)
shutil.rmtree(staging)

print(f"\n✅ Archive created: {zip_path}.zip")
zip_size_mb = os.path.getsize(f"{zip_path}.zip") / (1024 * 1024)
print(f"   Size: {zip_size_mb:.1f} MB")

# Trigger browser download
files.download(f"{zip_path}.zip")
print("\n📥 Download started! Extract the zip into your local project root.")

---
## 📋 After Downloading

1. Extract `fashionvision_v2_artifacts.zip` into your local project root:
   ```
   ML mini Project/
   ├── models_v2/
   │   ├── real_model.keras
   │   ├── class_names.json
   │   └── metadata.json
   ├── results_v2/
   │   ├── metrics.json
   │   ├── classification_report.json
   │   ├── confusion_matrix.png
   │   ├── misclassified_examples.png
   │   ├── training_curves.png
   │   └── history.json
   └── ...
   ```

2. Test locally:
   ```bash
   python scripts/check_real_inference.py --model-dir models_v2
   ```

3. Run the Streamlit app (once `app.py` is updated to support v2):
   ```bash
   streamlit run app.py
   ```